# T5-Base — NLP Robot Command Parser

Training, evaluation, ASR and HuRIC evaluation for the **t5-base** model.
Results are saved to `results/t5-base/` and checkpoints to `checkpoints/t5-base/final`.

Run this notebook independently — it loads data and trains from scratch.

## 0. Setup
Prepares the environment. The repository and the SCAN dataset are cloned from GitHub. Required dependencies are installed from requirements.txt. The configuration file (config.json) is loaded to set model and training parameters.

In [1]:
# Clone repo and move into it
!git clone https://github.com/PetraMicanovic/nlp-robot-command-parser.git
%cd nlp-robot-command-parser

Cloning into 'nlp-robot-command-parser'...
remote: Enumerating objects: 691, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 691 (delta 29), reused 42 (delta 14), pack-reused 633 (from 1)
Receiving objects: 100% (691/691), 404.49 KiB | 1.75 MiB/s, done.
Resolving deltas: 100% (387/387), done.
/content/nlp-robot-command-parser


In [2]:
!git clone https://github.com/brendenlake/SCAN.git data/scan

Cloning into 'data/scan'...
remote: Enumerating objects: 205, done.
remote: Total 205 (delta 0), reused 0 (delta 0), pack-reused 205 (from 1)
Receiving objects: 100% (205/205), 11.10 MiB | 13.95 MiB/s, done.
Resolving deltas: 100% (173/173), done.


In [3]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 16.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 76.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.28.0 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.


In [4]:
import json, sys, torch, random, numpy as np
sys.path.insert(0, '.')   # makes src/ importable

with open('config.json') as f:
    cfg = json.load(f)

SEED = cfg['training']['seed']
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print(f'Model  : {cfg["model_t5_base"]["name"]}')


Device : cuda
Model  : t5-base


## 1. Load data
Loads the SCAN dataset using the load_scan function. The data is split into training and test sets based on the configuration.

It can be loaded in either English or Serbian depending on the selected language (`cfg['data']['lang']`). When Serbian is selected, commands and actions are automatically translated.

Basic dataset informations are displayed.

In [ ]:
from src.data.load_data import load_scan

language = cfg['data']['lang'] # 'sr' or 'en'
split = cfg['data']['scan_split']

train_data, test_data = load_scan(
    split = split,
    base_path = cfg['data']['scan_base_path'],
    lang = language,
)

print(f'Language: {language}')
print(f'Train examples: {len(train_data)}')
print(f'Test examples: {len(test_data)}')
print(f'First example: {train_data[0]}')

### 1.1. Dataset statistics

In [ ]:
from src.data.translate_scan import print_stats

print_stats(train_data, test_data)

## 2. Preprocessing (tokenization)

This step prepares the dataset for sequence-to-sequence training with T5. A tokenizer is loaded based on the selected model, and the raw data is converted into Hugging Face `Dataset` format.

The dataset is then tokenized by adding a task-specific prefix, encoding commands and actions, and preparing labels for training. Tokenization is applied separately to the training and test splits using a `DatasetDict`.

In [ ]:
from src.data.preprocess import get_tokenizer, to_hf_dataset, tokenize_dataset
from datasets import DatasetDict

tokenizer_base = get_tokenizer(cfg['model_t5_base']['name'])

raw_dataset = DatasetDict({
    'train': to_hf_dataset(train_data),
    'test': to_hf_dataset(test_data),
})

tokenized_dataset_base = DatasetDict({
    split: tokenize_dataset(
        raw_dataset[split], tokenizer_base,
        prefix = cfg['model_t5_base']['prefix'],
        max_input_len = cfg['model_t5_base']['max_input_len'],
        max_target_len = cfg['model_t5_base']['max_target_len'],
    )
    for split in ('train', 'test')
})

print('Tokenization complete.')
print(tokenized_dataset_base)

##  3. Config for t5-base

In [11]:
model_cfg_base = cfg["model_t5_base"]
training_cfg_base = cfg["training_t5_base"]

print("Model:", model_cfg_base["name"])
print("Out dir:", training_cfg_base["output_dir"])

Model: t5-base
Out dir: checkpoints/t5-base


## 4. Training

Repeats the full training and evaluation pipeline with **t5-base**. Results are saved in a dedicated `results/t5-base/` folder so they never overwrite the t5-small results.

In [ ]:
# Load t5-base
from src.models.t5_model import load_model

model_base = load_model(model_cfg_base["name"], DEVICE)

In [ ]:
# Train t5-base
from src.training.trainer import build_trainer

trainer_base = build_trainer(
    model=model_base,
    tokenizer=tokenizer_base,
    tokenized_dataset=tokenized_dataset_base,
    cfg=cfg,
    model_key="model_t5_base",
    device_fp16=(DEVICE == "cuda"),
)

trainer_base.train()

In [ ]:
from src.training.trainer import get_checkpoint_dir

# Save t5-base checkpoint
SAVE_PATH = get_checkpoint_dir(cfg, "model_t5_base")
model_base.save_pretrained(SAVE_PATH)
tokenizer_base.save_pretrained(SAVE_PATH)
print(f"t5-base model saved to: {SAVE_PATH}")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
import shutil, os

drive_dest = f"/content/drive/MyDrive/nlp-robot-command-parser/{SAVE_PATH}"
os.makedirs(drive_dest, exist_ok=True)
shutil.copytree(SAVE_PATH, drive_dest, dirs_exist_ok=True)
print(f"Checkpoint copied to Google Drive: {drive_dest}")

## 5. Loading trained model

In [5]:
from src.training.trainer import get_checkpoint_dir
from transformers import T5ForConditionalGeneration, T5Tokenizer
import os

LOAD_PATH = get_checkpoint_dir(cfg, "model_t5_base")

try:
    from google.colab import drive

    drive.mount("/content/drive")
    LOAD_PATH = f"/content/drive/MyDrive/nlp-robot-command-parser/{LOAD_PATH}"
except ImportError:
    pass

if not os.path.exists(LOAD_PATH):
    raise FileNotFoundError(f"Model not found at: {LOAD_PATH}")

model_base = T5ForConditionalGeneration.from_pretrained(LOAD_PATH)
tokenizer_base = T5Tokenizer.from_pretrained(LOAD_PATH)
model_base = model_base.to(DEVICE)

print(f" Model loaded from: {LOAD_PATH}")
print(f" Device: {DEVICE}")

Mounted at /content/drive


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

 Model loaded from: /content/drive/MyDrive/nlp-robot-command-parser/checkpoints/t5-base/final
 Device: cuda


## 6. Model Evaluation

In [ ]:
from src.evaluation.evaluation import evaluate_model, print_exact_match
from src.evaluation.save_results import save_evaluation_results, copy_results_to_drive

# Build a minimal cfg dict scoped to t5-base
cfg_base = dict(cfg)
cfg_base["model"] = model_cfg_base

results_base = evaluate_model(
    test_data, model_base, tokenizer_base, cfg_base, DEVICE, n=cfg["data"]["n_eval"]
)
print_exact_match(results_base)

save_evaluation_results(
    results_base, split_name="test", cfg=cfg, model_key="model_t5_base"
)
copy_results_to_drive(cfg=cfg, model_key="model_t5_base")

In [ ]:
# Evaluate t5-base across SCAN splits
import pandas as pd
from src.data.load_data import load_scan
from src.evaluation.evaluation import evaluate_model
from src.evaluation.save_results import save_evaluation_results

splits_to_eval = ['simple', 'length', 'addprim_jump', 'addprim_turn_left', 'template_around_right','template_jump_around_right','template_opposite_right','template_right','filler_num0','filler_num1','filler_num2','filler_num3','fewshot_num8_rep1']

rows_base = []

for sp in splits_to_eval:
    _, test_sp = load_scan(
        split=sp, base_path=cfg["data"]["scan_base_path"], lang=cfg["data"]["lang"]
    )
    r = evaluate_model(
        test_sp, model_base, tokenizer_base, cfg_base, DEVICE, n=cfg["data"]["n_eval"]
    )
    rows_base.append({"split": sp, "exact_match": r["exact_match"]})
    save_evaluation_results(r, split_name=sp, cfg=cfg, model_key="model_t5_base")

df_base = pd.DataFrame(rows_base)
print(df_base.to_string(index=False))

## 7. Error analysis by command length

Evaluates model performance for different sequence lengths.
The results are grouped into `buckets`, printed to the console, and saved as a JSON file in the `results` directory.

In [ ]:
from src.evaluation.evaluation import analyse_by_length, print_length_analysis
from src.evaluation.save_results import save_length_analysis, copy_results_to_drive

buckets = analyse_by_length(
    test_data, model_base, tokenizer, cfg, DEVICE,
    n=cfg['data']['n_error_analysis']
)
print_length_analysis(buckets)

save_length_analysis(buckets, cfg=cfg, model_key='model_t5_base')
copy_results_to_drive(cfg=cfg, model_key='model_t5_base')

## 8. End-to-end pipeline evaluation

This section evaluates the complete T5-base pipeline from audio commands to action sequences. The pipeline is tested on randomly sampled SCAN examples using both normalized and raw audio inputs. Evaluation results, predictions and exact-match accuracy are saved and exported to Google Drive.

In [ ]:
from src.pipeline import RobotCommandPipeline
from src.evaluation.save_results import save_evaluation_results, copy_results_to_drive
from src.data.load_data import load_scan
import os, random

pipeline_base = RobotCommandPipeline(model_base, tokenizer_base, cfg_base, DEVICE)

random.seed(cfg["training"]["seed"])
_, test_data_for_asr = load_scan(
    split=cfg["data"]["scan_split"],
    base_path=cfg["data"]["scan_base_path"],
    lang=cfg["data"]["lang"],
)
asr_sample = random.sample(test_data_for_asr, cfg["asr"]["n_asr_samples"])

audio_dir_norm = "results/asr_audio"
audio_dir_raw = "results/asr_audio_raw"
if not os.path.exists(audio_dir_norm):
    audio_dir_norm = "/content/drive/MyDrive/nlp-robot-command-parser/results/asr_audio"
    audio_dir_raw = (
        "/content/drive/MyDrive/nlp-robot-command-parser/results/asr_audio_raw"
    )

correct_norm, correct_raw, per_example = 0, 0, []

for i in range(len(asr_sample)):
    filename = f"cmd_{i:04d}.mp3"
    audio_norm = os.path.join(audio_dir_norm, filename)
    audio_raw = os.path.join(audio_dir_raw, f"cmd_raw_{i:04d}.mp3")
    gold_actions = asr_sample[i]["actions"]

    result_norm = pipeline_base.run_from_audio(
        audio_norm, gold_actions=gold_actions, normalize=True
    )
    result_raw = pipeline_base.run_from_audio(
        audio_raw, gold_actions=gold_actions, normalize=False
    )

    correct_norm += result_norm["correct"]
    correct_raw += result_raw["correct"]
    per_example.append(
        {
            "command": asr_sample[i]["commands"],
            "gold_actions": gold_actions,
            "transcript_norm": result_norm["asr_transcript"],
            "predicted_norm": result_norm["predicted_actions"],
            "correct_norm": result_norm["correct"],
            "transcript_raw": result_raw["asr_transcript"],
            "predicted_raw": result_raw["predicted_actions"],
            "correct_raw": result_raw["correct"],
        }
    )

n = len(asr_sample)
print("End-to-end pipeline accuracy — t5-base (Audio -> Text -> Actions)")
print("=" * 60)
print(f"With normalization: {correct_norm / n:.2%}")
print(f"Without normalization: {correct_raw  / n:.2%}")

pipeline_results_base = {
    "with_normalization": {"exact_match": round(correct_norm / n, 4), "n_evaluated": n},
    "without_normalization": {
        "exact_match": round(correct_raw / n, 4),
        "n_evaluated": n,
    },
    "per_example": per_example,
}
save_evaluation_results(
    pipeline_results_base, split_name="pipeline", cfg=cfg, model_key="model_t5_base"
)
copy_results_to_drive(cfg=cfg, model_key="model_t5_base")

## 8.1. Voice demo (my own recordings)

Runs the trained t5-base pipeline on real human speech instead of gTTS-generated audio, to see how it performs on natural voice input.

Recordings are stored in `data/audio/my_voice_demo/`, with the expected commands and actions defined in `data/my_voice_commands.json`. Each recording is run both **with and without transcript normalization**, same as the 100-sample pipeline evaluation above

In [12]:
from src.pipeline import RobotCommandPipeline
from src.data.translate_scan import translate_actions
from src.evaluation.save_results import save_evaluation_results, copy_results_to_drive
import os
import json
import pandas as pd

cfg_base = dict(cfg)

pipeline_base = RobotCommandPipeline(model_base, tokenizer_base, cfg_base, DEVICE)

voice_dir = 'data/audio/my_voice_demo'
commands_json_path = 'data/my_voice_commands.json'

with open(commands_json_path, 'r', encoding='utf-8') as f:
    voice_commands = json.load(f)

print(f'Loaded {len(voice_commands)} voice command entries from {commands_json_path}')

per_example = []
correct_norm, correct_raw = 0, 0

for entry in voice_commands:
    audio_path = os.path.join(voice_dir, entry['audio_file'])

    if not os.path.exists(audio_path):
        print(f"Missing file: {audio_path}")
        continue

    expected_actions_en = entry.get('output')
    expected_actions_sr = translate_actions(expected_actions_en, cfg['data']['lang'])

    result_norm = pipeline_base.run_from_audio(audio_path, gold_actions=expected_actions_sr, normalize=True)
    result_raw = pipeline_base.run_from_audio(audio_path, gold_actions=expected_actions_sr, normalize=False)

    if result_norm.get('correct') is True:
        correct_norm += 1
    if result_raw.get('correct') is True:
        correct_raw += 1


    if expected_actions_sr:
        expected_display = expected_actions_sr
    else:
        expected_display = 'N/A'

    per_example.append({
        'File': entry['audio_file'],
        'Command': entry.get('input', ''),
        'Gold actions (sr)': expected_display,
        'Transcript (norm)': result_norm['asr_transcript'],
        'Predicted (norm)': result_norm['predicted_actions'],
        'Correct (norm)': result_norm.get('correct', 'N/A'),
        'Transcript (raw)': result_raw['asr_transcript'],
        'Predicted (raw)': result_raw['predicted_actions'],
        'Correct (raw)': result_raw.get('correct', 'N/A'),
    })

n = len(per_example)
if n:
    print(f"\nProcessed {n} recordings")
    print(f"With normalization: {correct_norm}/{n} ({correct_norm/n:.0%})")
    print(f"Without normalization: {correct_raw}/{n} ({correct_raw/n:.0%})")
else:
    print('\n No recordings processed.')

df_voice = pd.DataFrame(per_example)
display(df_voice.style.set_caption('Live voice demo results — t5-base'))

if n:
    exact_match_norm = round(correct_norm / n, 4)
    exact_match_raw = round(correct_raw / n, 4)
else:
    exact_match_norm = 0
    exact_match_raw = 0

voice_demo_results = {
    'with_normalization': {'exact_match': exact_match_norm, 'n_evaluated': n},
    'without_normalization': {'exact_match': exact_match_raw, 'n_evaluated': n},
    'per_example': per_example,
}

save_evaluation_results(voice_demo_results, split_name='voice_demo', cfg=cfg, model_key='model_t5_base')
copy_results_to_drive(cfg=cfg, model_key='model_t5_base')

Loaded 7 voice command entries from data/my_voice_commands.json
Loading Whisper model: small


100%|███████████████████████████████████████| 461M/461M [00:10<00:00, 45.0MiB/s]
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence.


Processed 7 recordings
With normalization: 5/7 (71%)
Without normalization: 3/7 (43%)


,File,Command,Gold actions (sr),Transcript (norm),Predicted (norm),Correct (norm),Transcript (raw),Predicted (raw),Correct (raw)
0,cmd_1.mp3,gledaj desno nakon sto skocis,I_SKOCI I_OKRENI_DESNO I_GLEDAJ,gledaj desno nakon sto skocis,I_SKOCI I_SKOCI I_OKRENI_DESNO I_GLEDAJ,False,gledaj desno nakon što skočiš.,I_SKOCI I_SKOCI I_OKRENI_DESNO I_GLEDAJ,False
1,cmd_2.mp3,hodaj lijevo,I_OKRENI_LIJEVO I_HODAJ,hodaj lijevo,I_OKRENI_LIJEVO I_HODAJ,True,hoda i ljevo.,I_HODA I_OKRENI_LIJEVO,False
2,cmd_3.mp3,hodaj lijevo okolo,I_OKRENI_LIJEVO I_HODAJ I_OKRENI_LIJEVO I_HODAJ I_OKRENI_LIJEVO I_HODAJ I_OKRENI_LIJEVO I_HODAJ,hodaj lijevo okolo,I_OKRENI_LIJEVO I_HODAJ I_OKRENI_LIJEVO I_HODAJ I_OKRENI_LIJEVO I_HODAJ I_OKRENI_LIJEVO I_HODAJ,True,hoda i lievo okolo.,I_HODA I_OKRENI_LIVO I_OKRENI_LIVO I_OKRENI_LIVO I_OKRENI_LIVO I_OKRENI_LIVO,False
3,cmd_4.mp3,hodaj suprotno od desno,I_OKRENI_DESNO I_OKRENI_DESNO I_HODAJ,hodaj suprotno od desno,I_OKRENI_DESNO I_OKRENI_DESNO I_HODAJ I_OKRENI_DESNO I_OKRENI_DESNO I_HODAJ,False,hoda i suprotno od desno.,I_HODA I_OKRENI_DESNO I_OKRENI_DESNO I_OKRENI_DESNO,False
4,cmd_5.mp3,okreni se desno tri puta i gledaj,I_OKRENI_DESNO I_OKRENI_DESNO I_OKRENI_DESNO I_GLEDAJ,pokreni se desno tri puta i gledaj,I_OKRENI_DESNO I_OKRENI_DESNO I_OKRENI_DESNO I_GLEDAJ,True,pokreni se desno 3 puta i gledaj.,I_OKRENI_DESNO I_OKRENI_DESNO I_OKRENI_DESNO I_GLEDAJ,True
5,cmd_6.mp3,skoci tri puta,I_SKOCI I_SKOCI I_SKOCI,skoci tri puta,I_SKOCI I_SKOCI I_SKOCI,True,skoči tri puta.,I_SKOCI I_SKOCI I_SKOCI,True
6,cmd_7.mp3,trci dva puta i skoci,I_TRCI I_TRCI I_SKOCI,traci dva puta i skoci,I_TRCI I_TRCI I_SKOCI,True,traci dva puta i skoći.,I_TRCI I_TRCI I_SKOCI,True


Evaluation results saved to: results/t5-base/evaluation_voice_demo.json
Results folder copied to Google Drive: /content/drive/MyDrive/nlp-robot-command-parser/results/t5-base


## 9. HuRIC evaluation

In [ ]:
import json as _json
import pandas as pd
from src.evaluation.evaluation import evaluate_model, print_exact_match
from src.evaluation.save_results import save_evaluation_results, copy_results_to_drive
from src.models.t5_model import predict
from src.data.translate_scan import translate_actions

with open(
    "data/sr_huric_scan_generalization_subset_18.json", "r", encoding="utf-8"
) as f:
    huric_raw = _json.load(f)

huric_test = [
    {"commands": ex["input"], "actions": translate_actions(ex["output"], "sr")}
    for ex in huric_raw
]

results_huric_base = evaluate_model(
    huric_test, model_base, tokenizer_base, cfg_base, DEVICE, n=len(huric_test)
)
print("HuRIC Evaluation — t5-base:")
print_exact_match(results_huric_base)

rows_huric_base = []
for ex in huric_test:
    pred = predict(
        ex["commands"],
        model_base,
        tokenizer_base,
        prefix=model_cfg_base["prefix"],
        max_input_len=model_cfg_base["max_input_len"],
        max_target_len=model_cfg_base["max_target_len"],
        device=DEVICE,
        num_beams=model_cfg_base["num_beams"],
    ).strip()
    rows_huric_base.append(
        {
            "Command": ex["commands"],
            "Expected": ex["actions"],
            "Predicted": pred,
            "Correct": "correct" if pred == ex["actions"].strip() else "incorrect",
        }
    )

df_huric_base = pd.DataFrame(rows_huric_base)
df_huric_base.index += 1
display(
    df_huric_base.style.set_caption(
        f"HuRIC — t5-base (Accuracy: {results_huric_base['exact_match']:.2%})"
    )
)

save_evaluation_results(
    {
        "exact_match": results_huric_base["exact_match"],
        "n_evaluated": results_huric_base["n_evaluated"],
        "per_example": rows_huric_base,
    },
    split_name="huric",
    cfg=cfg,
    model_key="model_t5_base",
)
copy_results_to_drive(cfg=cfg, model_key="model_t5_base")